# Chapter 8 — RAG Generation
## Topic 4: Hallucination as a Measured Failure Mode

**Run cells in order.**

- Module 1: Setup — claims (some grounded, some not) + a mock LLM-judge client
- Module 2: Tier 1 — embedding-similarity coarse filter
- Module 3: Tier 2 — simulated LLM-as-judge escalation (mock client, offline-safe)
- Module 4: Full two-tier pipeline + cost/volume analysis

**Install:** `pip install sentence-transformers numpy`
(Real Tier 2 requires `client = anthropic.Anthropic(...)` — a MockClient
stands in here so this notebook runs fully offline; swap in the real
client for production use, shown in a comment.)


## Module 1: Setup — Claims + Mock LLM-Judge Client

**Requires:** nothing standalone

In [ ]:
"""
MODULE 1: Setup

Claims spanning the full spectrum: clearly supported, clearly contradicted,
and subtly wrong (same vocabulary, different meaning -- the specific case
that defeats pure embedding-similarity checking, per the theory).

MockClient simulates client.messages.create() for the LLM-as-judge tier
WITHOUT a real API call -- replace with `anthropic.Anthropic(api_key=...)`
and `claude-haiku-4-5-20251001` for production (shown in Module 3's comment).
"""

CONTEXT_MAP = {
    "01_FD_FAQ.pdf": "Premature withdrawal of FD incurs a 1 percent penalty on the applicable interest rate.",
    "04_FD_Policy_Reference.pdf": "Senior citizens receive an additional 0.5 percent interest on all Fixed Deposit tenures.",
}

CLAIMS = [
    {"text": "The penalty for premature withdrawal is 1 percent.",
     "cited_source": "01_FD_FAQ.pdf", "ground_truth": "SUPPORTED"},
    {"text": "The penalty for premature withdrawal is 5 percent.",
     "cited_source": "01_FD_FAQ.pdf", "ground_truth": "CONTRADICTED"},
    {"text": "There is no penalty at all for premature withdrawal.",
     "cited_source": "01_FD_FAQ.pdf", "ground_truth": "CONTRADICTED"},  # subtle -- same vocab, opposite meaning
    {"text": "Senior citizens receive extra interest on their FDs.",
     "cited_source": "04_FD_Policy_Reference.pdf", "ground_truth": "SUPPORTED"},
]


class MockClient:
    """Simulates client.messages.create() for the LLM-as-judge tier.
    Uses simple keyword-based logic to approximate what a real judge call
    would return -- NOT a substitute for real judge quality, purely for
    offline pipeline-mechanics verification."""

    class MockResponse:
        def __init__(self, text):
            self.content = [type("Block", (), {"text": text})()]

    class Messages:
        def create(self, model, max_tokens, messages):
            prompt = messages[0]["content"]
            # crude simulation: look for negation words as a "contradiction" signal
            if "no penalty" in prompt.lower() or "not " in prompt.lower():
                verdict = "NO"
            elif any(str(n) in prompt for n in [5, 10, 20]) and "1 percent" in prompt:
                verdict = "NO"  # number mismatch detected in the simulated prompt
            else:
                verdict = "YES"
            return MockClient.MockResponse(verdict)

    def __init__(self):
        self.messages = MockClient.Messages()


mock_client = MockClient()
MODEL_ID = "claude-haiku-4-5-20251001"  # same model used throughout this project

print("Claims to verify:")
for i, c in enumerate(CLAIMS, start=1):
    print(f"  {i}. \'{c['text']}\' [cited: {c['cited_source']}] (ground truth: {c['ground_truth']})")

print("\nModule 1 complete. Run Module 2.")


## Module 2: Tier 1 — Embedding-Similarity Coarse Filter

**Requires:** Module 1

In [ ]:
"""
MODULE 2: Tier 1 -- Embedding Similarity

Cheap, fast, run on EVERY claim. Flags claims for Tier 2 escalation.
Demonstrates the theory's specific warning: embedding similarity CANNOT
reliably distinguish support from contradiction when vocabulary overlaps.
"""

import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading embedding model (may take a moment on first run)...")
embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")


def check_entailment_embedding(claim_text: str, source_text: str, embed_model,
                                threshold: float = 0.5) -> dict:
    claim_vec = embed_model.encode(claim_text, normalize_embeddings=True, show_progress_bar=False)
    source_vec = embed_model.encode(source_text, normalize_embeddings=True, show_progress_bar=False)
    similarity = float(np.dot(claim_vec, source_vec))
    return {"similarity": similarity, "flagged": similarity < threshold}


print("\nTier 1 results:\n")
tier1_results = []
for i, c in enumerate(CLAIMS, start=1):
    source_text = CONTEXT_MAP[c["cited_source"]]
    result = check_entailment_embedding(c["text"], source_text, embed_model)
    tier1_results.append(result)
    print(f"Claim {i}: \'{c['text']}\'")
    print(f"  Similarity: {result['similarity']:.4f} | Flagged for Tier 2: {result['flagged']}")
    print(f"  Ground truth: {c['ground_truth']}")
    print()

print("Key observation to watch for: Claim 3 (\'no penalty at all\' -- directly")
print("CONTRADICTS the source) shares heavy vocabulary with the source text")
print("(\'penalty\', \'premature withdrawal\') and may score a DECEPTIVELY HIGH")
print("similarity despite being factually opposite -- exactly the failure mode")
print("the theory warns about. Watch its similarity score above.")

print("\nModule 2 complete. Run Module 3.")


## Module 3: Tier 2 — LLM-as-Judge Escalation

**Requires:** Module 1, Module 2

In [ ]:
"""
MODULE 3: Tier 2 -- LLM-as-Judge

Runs ONLY on claims Tier 1 flagged. Uses the MockClient from Module 1.

REAL PRODUCTION CODE (uncomment and use with a real API key):

    import anthropic
    client = anthropic.Anthropic(api_key=os.getenv("anthropic_api_key"))
    # then call check_entailment_llm_judge(claim_text, source_text, client, MODEL_ID)
    # exactly as below -- the function signature is identical for mock and real.
"""

def check_entailment_llm_judge(claim_text: str, source_text: str, client, model_id: str) -> dict:
    judge_prompt = f"""Source passage: "{source_text}"
Claim: "{claim_text}"

Does the source passage support this claim? Answer with exactly one word: YES, NO, or PARTIAL."""
    response = client.messages.create(
        model=model_id, max_tokens=10,
        messages=[{"role": "user", "content": judge_prompt}],
    )
    verdict = response.content[0].text.strip().upper()
    return {"verdict": verdict, "entailed": verdict == "YES"}


print("Tier 2 escalation (only for Tier 1-flagged claims):\n")
tier2_results = {}
for i, (c, t1) in enumerate(zip(CLAIMS, tier1_results), start=1):
    if t1["flagged"]:
        source_text = CONTEXT_MAP[c["cited_source"]]
        t2 = check_entailment_llm_judge(c["text"], source_text, mock_client, MODEL_ID)
        tier2_results[i] = t2
        print(f"Claim {i}: \'{c['text']}\'")
        print(f"  Tier 2 verdict: {t2['verdict']} | Ground truth: {c['ground_truth']}")
        match = (t2["verdict"] == "NO" and c["ground_truth"] == "CONTRADICTED") or \
                (t2["verdict"] == "YES" and c["ground_truth"] == "SUPPORTED")
        print(f"  Matches ground truth: {match}")
        print()
    else:
        print(f"Claim {i}: not flagged by Tier 1 -- Tier 2 skipped (cost saved)\n")

print("Module 3 complete. Run Module 4.")


## Module 4: Full Two-Tier Pipeline + Cost/Volume Analysis

**Requires:** Module 1, Module 2, Module 3

In [ ]:
"""
MODULE 4: Full Pipeline + Cost Analysis

Ties Tier 1 and Tier 2 together into a final per-claim verdict, then
projects the cost implication of tiering vs. running Tier 2 on everything,
at this project's production volume.
"""

def evaluate_answer_faithfulness(claims: list, context_map: dict, embed_model,
                                  client, model_id: str, embedding_threshold: float = 0.5) -> dict:
    results = []
    tier2_calls = 0
    for claim in claims:
        source_text = context_map.get(claim["cited_source"], "")
        tier1 = check_entailment_embedding(claim["text"], source_text, embed_model, embedding_threshold)
        result = {"claim": claim["text"], "tier1": tier1}
        if tier1["flagged"]:
            tier2 = check_entailment_llm_judge(claim["text"], source_text, client, model_id)
            result["tier2"] = tier2
            tier2_calls += 1
        results.append(result)

    unsupported = [r for r in results
                   if r.get("tier2", {}).get("verdict") == "NO"
                   or (r["tier1"]["flagged"] and "tier2" not in r)]
    return {
        "all_supported": len(unsupported) == 0,
        "claim_results": results,
        "unsupported": unsupported,
        "tier2_calls_made": tier2_calls,
        "total_claims": len(claims),
    }


final = evaluate_answer_faithfulness(CLAIMS, CONTEXT_MAP, embed_model, mock_client, MODEL_ID)

print(f"All claims supported: {final['all_supported']}")
print(f"Unsupported claims found: {len(final['unsupported'])}")
for u in final["unsupported"]:
    print(f"  UNSUPPORTED: \'{u['claim']}\'")

print(f"\nTier 2 calls made: {final['tier2_calls_made']}/{final['total_claims']} claims "
      f"({final['tier2_calls_made']/final['total_claims']:.0%} escalation rate)")

# -----------------------------------------------------------------------
# Cost projection at production volume
# -----------------------------------------------------------------------
print("\n" + "=" * 65)
print("COST PROJECTION AT PRODUCTION VOLUME (8,000-12,000 emails/day)")
print("=" * 65)

avg_claims_per_answer = 3       # illustrative
escalation_rate = final["tier2_calls_made"] / final["total_claims"]
daily_emails = 10_000            # midpoint of range

total_claims_per_day = daily_emails * avg_claims_per_answer
tier1_calls_per_day = total_claims_per_day * 2  # 2 embed calls per claim (claim + source)
tier2_calls_per_day_tiered = total_claims_per_day * escalation_rate
tier2_calls_per_day_untiered = total_claims_per_day  # if EVERY claim got Tier 2

print(f"Estimated claims/day: {total_claims_per_day:,} (at {avg_claims_per_answer} claims/answer)")
print(f"Tier 1 embedding calls/day: {tier1_calls_per_day:,.0f} (cheap, local, no API cost)")
print(f"Tier 2 LLM-judge calls/day WITH tiering: {tier2_calls_per_day_tiered:,.0f} "
      f"({escalation_rate:.0%} escalation rate)")
print(f"Tier 2 LLM-judge calls/day WITHOUT tiering (naive): {tier2_calls_per_day_untiered:,.0f}")
print(f"\nTiering reduces expensive Tier 2 volume by "
      f"{(1 - tier2_calls_per_day_tiered/tier2_calls_per_day_untiered):.0%} "
      f"in this illustrative scenario.")

print("\n" + "=" * 65)
print("CHAPTER 8 TOPIC 4 SUMMARY")
print("=" * 65)
print("""
Two-tier hallucination detection: Tier 1 (embedding similarity) runs on
  every claim, cheap but unreliable at distinguishing support from
  contradiction when vocabulary overlaps -- demonstrated by Claim 3
  ("no penalty at all") in Module 2.
Tier 2 (LLM-as-judge) runs ONLY on Tier 1-flagged claims -- reliable but
  expensive, kept proportional to actual risk via tiering.
At this project's production volume, tiering meaningfully reduces the
  volume of expensive Tier 2 calls compared to judging every claim.
Distinct from Topic 3's relevance/correctness checks -- this pipeline
  specifically targets FAITHFULNESS (does the claim match its cited source).

Next: Topic 5 -- Streaming Responses in a RAG Pipeline
""")
